### LangGraph

#### Core Concepts
- [State](#state)
- [Nodes](#node)
- [Edges](#edges)
- [StateGraph](#state_graph)
- [LLM Calls](#llm-call)
- [Messages](#messages)
- [Commands](#commands)
- [Render Graph](#render)

#### Advanced
- [Memory](#memory)
- [Tools](#tools)
- [Structured Output](#structured_output)
- [Interrupts](#hitl)
- [RAG](#rag)

#### Managing State <a id='state'></a>
- State(Data): All nodes share the same **State object** which can be a *Python TypedDict*, *dataclass*, or a *Pydantic BaseModel*
    * reducer - automatically called to combine this response with previous responses. 
        * `operator.add` - to accumulate results from parallel branches. From python `import operator `
        * `add_message` - from `langgraph.graph`
    * ```Annotated[list[str], operator.add]``` - State filed definition with data type and metadata (reducer function)

In [ ]:
from typing import Annotated, Any, List, Optional, TypedDict
from langgraph.graph import add_messages

class AgentState(TypedDict):
    email_content: str
    messages: Annotated[List[Any], add_messages]
    success_criteria: str
    feedback_on_work: Optional[str]
    success_criteria_met: bool
    classification: EmailClassification | None
    draft_response: str | None

#### Node <a id='node'></a>

- Nodes(Functions): Defined as simple **Python functions** that receive State as input and return **new** State object with updated fields value
    * Functional Logic
    * LLM Calls
    * RAG or MCP invocation

In [ ]:
def node_a(state: State) -> State:
    print(f"Adding 'A' to {state['nlist']}")
    return(State(nlist = ["A"]))

#### State Graph <a id='state_graph'></a>

- StateGraph
    * add_node
    * add_edge
    * compile

In [ ]:
from langgraph.graph import StateGraph, START, END

# Start the Graph Builder with State class
graph_builder = StateGraph(State)

# Add nodes to the graph
graph_builder.add_node(node_a)
graph_builder.add_node(node_b)

# Add edges to the graph
graph_builder.add_edge(START, node_a)
graph_builder.add_edge(node_a, END)

# Compile the graph to get an executable graph object
graph = graph_builder.compile()

#### Structured Output <a id='structured_output'></a>

- Structured Outputs
    * Create (Pydantic/TypedDict) class to describe desired LLM schema output.
    * Send this class to LLM, invocation result now will be formatted with this stuctured output.

In [ ]:
from typing import Literal, TypedDict

class EmailClassification(TypedDict):
    intent: Literal["question", "bug", "billing", "feature", "complex"]
    urgency: Literal["low", "medium", "high", "critical"]
    topic: str
    summary: str

    # Create structured LLM that returns EmailClassification dict
    structured_llm = llm.with_structured_output(EmailClassification)

#### Messages <a id='messages'></a>

- Messages
    * HymanMessage
    * AIMessage
    * SystemMessage
    * ToolMessage

#### Commands <a id='commands'></a>

- In LangGraph, Command is a special object that controls the execution of the graph.
- With Command node decides routing itself and normal edges are ignored

In [ ]:
from langgraph.types import Command

# Example Node function that uses Command to control flow
if needs_tool:
    return Command(goto="tool_node")

# If the node is terminal and we are done, go to END
if done:
    return Command(goto=END)

# Command update state and graph immediately goes to the next Node
if feedback:
    return Command(
        update = State(messages = [new_message]),
        goto = [next_node]
    )

#### Tools <a id='tools'></a>

- tools
    * Custom Tools 
    * Community Tools
    * ToolNode

In [ ]:
from langchain_core.tools import tool
from langgraph.prebuilt import ToolNode
from langgraph.graph import END
from langgraph.prebuilt import ToolNode
from langgraph.types import Command

#Step 1: Define some tools that the agent can use. 
# These can be any Python functions that perform specific tasks. 
# For example, let's define two simple tools: one for adding numbers and another for subtracting numbers.

@tool
def add_numbers(num1: int, num2: int) -> int:
    """Tool function to add two numbers"""
    return num1 + num2

@tool
def sub_numbers(num1: int, num2: int) -> int:
    """Tool function to subtract two numbers"""
    return num1 - num2

# Step 2: Create a list of tools that the agent can use and bind them to the LLM.
tools = [add_numbers, sub_numbers]

#Step3: Bind the tools to the LLM so that it can call them when needed. 
# This allows the LLM to use these tools as part of its reasoning process.
llm = llm.bind_tools(tools)

# Step 4: Create a ToolNode that will handle tool calls from the LLM.
tool_node = ToolNode(tools=tools)

# Step 5: Create a router node that will determine which node to route to based on the presence of tool calls in the last message from the LLM.
def router(state: State) -> Command[Literal["tools_node", END]]:
    """Node function to determine which node to route to based on the current state."""
    last_message = state['messages'][-1]        # Get the last message from the message history
    if last_message.tool_calls:                 # Check if the last message contains any tool calls
        next_node = "tools_node"                # If it does, route to the tool node
    else:
        next_node = END
    return Command(goto=next_node)  # Return the command to route to the appropriate node

# Step 6: Add the ToolNode and the router node to the graph, and connect them appropriately.
graph.add_node("tools_node", tool_node)
graph.add_node("router", router)

#### LLM Call <a id='llm_call'></a>

- LLM Calls
    * tools binding
    * structured outputs

In [ ]:
llm = ChatOpenAI(model="gpt-5-mini")
structured_llm = llm.with_structured_output(EmailClassification)
llm_with_tools = llm.bind_tools(all_tools)

#### Edges <a id='edges'></a>

- Edges(Control Flow): Placed between Nodes. Defines how task are connected and executed.
    * Static Edges - Serial (execution flow movet through one to another Nodes sequentially) Parallel (enable parallel execution when one node has multiple outgoing edges)
    * Conditional Edges - Conditional (enable dynamic routing where the next node to execute depends on the current state) Map-Reducer
        * `Command` approach - to return both state updates and next Node routing decisions 
        * `add_conditional_edges`

In [ ]:
# Command approach
def node_a(state: State) -> Command[Literal["b", "c", END]]:
    select = state["messages"][-1] # Get the last message from the state
    if select == "b":
        next_node = "b"
    elif select == "c":
        next_node = "c"
    elif select == "q":
        next_node = END
    else:
        next_node = END
# Command allows us to specify both the state updates and the next node to transition to based on the input state
    return Command(
        update = State(nlist = [select]),
        goto = [next_node]
    )

# Router node approach
# Step 1: Define a worker router function that takes the state as input and returns the name of the next node to route to
def worker_router(state: State) -> str:
    last_message = state["messages"][-1]
    if hasattr(last_message, "tool_calls") and last_message.tool_calls:
        return "tools"
    else:
        return "evaluator"

def route_based_on_evaluation(state: State) -> str:
    if state["success_criteria_met"] or state["user_input_needed"]:
        return "END"
    else:
        return "worker"

# Step 2: Add conditional edges in the graph builder that use the router function to determine the next node
graph_builder.add_conditional_edges("worker", worker_router, {"tools": "tools", "evaluator": "evaluator"})
graph_builder.add_edge("tools", "worker")
graph_builder.add_conditional_edges("evaluator", route_based_on_evaluation, {"worker": "worker", "END": END})
graph_builder.add_edge(START, "worker")

#### Checkpointers / Memory <a id='memory'></a>

- Checkpointers / Memory: your graph can maintain conversation history or accumulate results over time.
    * InMemorySaver()
    * PostgersSaver()
    * SqliteSaver()
    * config = {"configurable": {"thread_id":"1"}} - accesses the same persisted state within the same thread
    * Snapshots

In [ ]:
from langgraph.checkpoint.memory import InMemorySaver

# define checkpointer and config
memory = InMemorySaver()
config = {"configurable": {"thread_id": "1"}}

# Compile the graph with the checkpointer
graph = builder.compile(checkpointer=memory)

# Invoke with input state and config
result = graph.invoke(input_state, config)

In [ ]:
import sqlite3
from langgraph.checkpoint.sqlite import SqliteSaver

db_path = "memory.db"
conn = sqlite3.connect(db_path, check_same_thread=False)
sql_memory = SqliteSaver(conn)
config = {"configurable": {"thread_id": "2"}}

# Compile the graph with the checkpointer
graph = builder.compile(checkpointer=sql_memory)

# Invoke with input state and config
result = graph.invoke(input_state, config)


#### HITL / Interrupts <a id='hitl'></a>

- HITL / Interrupts: function pauses graph execution and waits for human input before continuing This enables human-in-the-loop workflows where an admin can review unexpected situations and decide how to proceed. Interrupts require a checkpointer to save state between pauses.
    * Define **Interrupt condition** and **handler** within Node
    * Catch interrupt keyword in LLM response and resume process with `Command (resume = "")`

In [ ]:

# Step 1:  Define the node function with interrupt handling

def node_a(state: State) -> Command[Literal["b", "c", END]]:
    print("Entered 'a' node")
    select = state["nlist"][-1]
    if select == "b":
        next_node = "b"
    elif select == "c":
        next_node = "c"
    elif select == "q":
        next_node = END
    else:
        # If the input is unexpected, trigger an interrupt and ask the user for input to continue or quit.
        # The interrupt message will be printed, and the user will be prompted to enter a response.
        admin = interrupt(f"Unexpected input '{select}'")
        print(admin)
        if admin == "continue":
            next_node = "b"
        else:
            next_node = END
            select = "q"
            
    return Command(
        update = State(nlist = [select]),
        goto = next_node
    )

#Step 2: Invoke with input state and config
result = graph.invoke(input_state, config)

# Check for interrupt and handle it 
# In this example, if the user inputs an unexpected value, it will trigger an interrupt. 
# The interrupt message will be printed, and the user will be prompted to enter a response. 
# The response will then be used to resume the graph execution.

#Step 3: Check for interrupt in LLM Response and handle it
if '__interrupt__' in result:
    print(f"Interrupt:{result}")
    msg = result['__interrupt__'][-1].value
    print(msg)
    human = input(f"\n{msg}: ")
    human_response = Command(
        resume = human
    )
    result = graph.invoke(human_response, config)

#### RAG <a id='rag'></a>

- RAG
    * Embeddings are used to convert text into vector representations that can be stored in a vector database for efficient retrieval. 
    * Chunking is the process of breaking down large documents into smaller, more manageable pieces (chunks) that can be processed by the agent.
    * The retriever is a tool that allows the agent to retrieve relevant document chunks from the vector database based on a query.

In [ ]:
import os
from langchain_community.tools import tool
from langchain_openai import OpenAIEmbeddings
from langchain_community.document_loaders import PyPDFLoader
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_chroma import Chroma

    # Step 1: Import Enbedding model (should be the same as the one used to create the vector store and the one used in the graph)
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

    # Step 2: Load the Document
pdf_path = "Contract.pdf"
if not os.path.exists(pdf_path):
    raise FileNotFoundError(f"The specified PDF document was not found at: {pdf_path}")

pdf_loader = PyPDFLoader(pdf_path)
try:
    pages = pdf_loader.load()
    print(f"Successfully loaded {len(pages)} pages from the PDF document.")
except Exception as e:
    raise RuntimeError(f"An error occurred while loading the PDF document: {e}")

    # Step 3: Split the document into chunks (if needed) and create embeddings for each chunk
text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
pages_split = text_splitter.split_documents(pages)

    # Step 4: Set up and create the vector store
persist_directory = "chroma_db"
if not os.path.exists(persist_directory):
    os.makedirs(persist_directory)
collection_name = "vector_store"

# Actually creating the Chroma vector database and handling any potential errors that may occur during the creation process.
try:
    vector_store = Chroma.from_documents(
        documents=pages_split,
        embedding=embeddings,
        persist_directory=persist_directory,
        collection_name=collection_name
    )
    print(f"Successfully created the Chroma vector database with {len(pages_split)} document chunks.")
except Exception as e:
    raise RuntimeError(f"An error occurred while creating the Chroma vector database: {e}")

    # Step 5: Create Retriever from the vector store
retriever = vector_store.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 5}  # Retrieve the top 5 most similar document chunks. Default is 4
)

    # Step 6: Create Retriever Tool for the agent to interact with the vector database and retrieve relevant information based on user queries.
@tool
def retrieve_tool(query: str) -> str:
    """Tool for retrieving relevant document chunks based on a query."""
    
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant information in the document found."
    results = []
    for i, doc in enumerate(docs):
        results.append(f"Chunk {i+1}:\n{doc.page_content}\n")
    return "\n\n".join(results)

tools = [retrieve_tool]

    # Step 7: Bind the tools to the LLM and provide the LLM with the necessary context to use the tools effectively.

#### Render Graph <a id='render'></a>

In [ ]:
from IPython.display import Image, display
display(Image(compiled_graph.get_graph().draw_mermaid_png()))